In [1]:
import networkx as nx
from parser import *
import matplotlib.pyplot as plt

In [2]:
pomdp = POMDP("iff.POMDP")

In [3]:
pomdp.get_num_distances()

10

In [4]:
pomdp.find_absorbing_states()

{0, 1, 2, 3}

In [5]:
pomdp.enemy_states_d

{20.0: {54, 55, 56, 57, 58},
 16.2: {59, 60, 61, 62, 63},
 12.8: {64, 65, 66, 67, 68},
 9.8: {69, 70, 71, 72, 73},
 7.2: {74, 75, 76, 77, 78},
 5.0: {79, 80, 81, 82, 83},
 3.2: {84, 85, 86, 87, 88},
 1.8: {89, 90, 91, 92, 93},
 0.8: {94, 95, 96, 97, 98},
 0.2: {99, 100, 101, 102, 103}}

In [6]:
pomdp.friend_states_d

{-30.0: {4, 5, 6, 7, 8},
 -24.3: {9, 10, 11, 12, 13},
 -19.2: {14, 15, 16, 17, 18},
 -14.7: {19, 20, 21, 22, 23},
 -10.8: {24, 25, 26, 27, 28},
 -7.5: {29, 30, 31, 32, 33},
 -4.8: {34, 35, 36, 37, 38},
 -2.7: {39, 40, 41, 42, 43},
 -1.2: {44, 45, 46, 47, 48},
 -0.3: {49, 50, 51, 52, 53}}

## Analysis visibility changes

In [7]:
rewards_d = pomdp.friend_states_d
rewards_p = list(rewards_d.keys())
distance_d = {}

for (index, r_p) in enumerate(rewards_p):
    distance_d[index] = set()
    for v in rewards_d[r_p]:
        distance_d[index].add(v)
distance_d

{0: {4, 5, 6, 7, 8},
 1: {9, 10, 11, 12, 13},
 2: {14, 15, 16, 17, 18},
 3: {19, 20, 21, 22, 23},
 4: {24, 25, 26, 27, 28},
 5: {29, 30, 31, 32, 33},
 6: {34, 35, 36, 37, 38},
 7: {39, 40, 41, 42, 43},
 8: {44, 45, 46, 47, 48},
 9: {49, 50, 51, 52, 53}}

In [8]:
def get_action_adjl(vertices: Set[int], action: int) -> Dict[int, Set[int]]:
    adj_l_nop = {}
    for v in vertices:
        adj_l_nop[v] = set()
    for v in vertices:
            for vv in vertices:
                prob = pomdp.get_trans(v, action, vv)
                if prob > 0:
                    adj_l_nop[v].add(vv)

    return adj_l_nop

def check_other_adj_list(vertices: Set[int]):
    adj_list = None
    for action in pomdp.actions:
        if action != 2:
            temp = get_action_adjl(vertices, action)
            if adj_list is None:
                adj_list = temp

            assert(adj_list == temp)

def sort_by_visibility(vertices: Set[int]) -> List[int]:
    # returns a list in which the first vertex is the one with less visibility
    ## LEAST visibility vertex
    # to detect the vertex with least visibility we can see the transitions of the no-op action
    # the no-op action can only descrease visibility, but the lowest level of visibility cannot decrease further
    adj_l_nop = get_action_adjl(vertices, 2)
    count_abs = 0
    least_vis = None
    for (v, values) in adj_l_nop.items():
        if len(values) == 1:
            least_vis = v
            count_abs += 1
        else:
            assert(len(values) == 2)
    assert(count_abs == 1)

    # for all other action the adjancency lists should be the same
    check_other_adj_list(vertices)
    result = [least_vis]

    visited = set()
    visited.add(least_vis)

    adj_l_attack = get_action_adjl(vertices, 3)

    queue = [least_vis]

    while len(queue) > 0:
        current = queue.pop(0)
        assert(0 < len(adj_l_attack[current]) < 3)
        assert (current in adj_l_attack[current])

        for v in adj_l_attack[current]:
            if v not in visited:
                visited.add(v)
                result.append(v)
                queue.append(v)

    assert(len(adj_l_attack[result[-1]]) == 1)
    return result


In [9]:
for (d, vertices) in distance_d.items():
    if d > 0:
        print(d, sort_by_visibility(vertices))

1 [9, 10, 11, 12, 13]
2 [14, 15, 16, 17, 18]
3 [19, 20, 21, 22, 23]
4 [24, 25, 26, 27, 28]
5 [29, 30, 31, 32, 33]
6 [34, 35, 36, 37, 38]
7 [39, 40, 41, 42, 43]
8 [44, 45, 46, 47, 48]
9 [49, 50, 51, 52, 53]


## Can the airplane go backwards?

In [10]:
rewards_p = list(pomdp.enemy_states_d.keys())
rewards_p

[20.0, 16.2, 12.8, 9.8, 7.2, 5.0, 3.2, 1.8, 0.8, 0.2]

In [11]:
rewards_n = list(pomdp.friend_states_d.keys())
rewards_n

[-30.0, -24.3, -19.2, -14.7, -10.8, -7.5, -4.8, -2.7, -1.2, -0.3]

In [12]:
distance_d = {}

for (index, (r_p, r_n)) in enumerate(zip(rewards_p, rewards_n)):
    distance_d[index] = set()
    for v in pomdp.enemy_states_d[r_p]:
        distance_d[index].add(v)
    for v in pomdp.friend_states_d[r_n]:
        distance_d[index].add(v)

In [13]:
def can_it_go_backwards(v, d) -> bool:
    reach = pomdp.get_connected_vertices(v)
    for (d_, vertices_) in distance_d.items():
        if d_ > d:
            for v_ in vertices_:
                if v_ in reach:
                    return True

    return False

for (d, vertices) in distance_d.items():
    for v in vertices:
        if can_it_go_backwards(v, d):
            print("can go backwards", d, v)

## Modify the original POMDP as follows:
- For the friend there is only one level of visibility
- visibility can never decrease

In [14]:
new_states = set()
new_actions = pomdp.actions
new_trans_f = {} # trans_f: Dict[int, Dict[int, Dict[int, float]]]
new_obs_f = {} # action, v_to, obs -> prob
new_observations = pomdp.observations
new_rewards = {} # vertex, action -> reward

In [15]:
# keep only one visibility level from friend plane
rewards_d = pomdp.friend_states_d
rewards_p = list(rewards_d.keys())
distance_d = {}

new_states = set()
for (index, r_p) in enumerate(rewards_p):
    distance_d[index] = set()
    for v in rewards_d[r_p]:
        distance_d[index].add(v)

for (d, vertices) in distance_d.items():
    new_states.add(min(vertices))

for (r, vertices) in pomdp.enemy_states_d.items():
    for v in vertices:
        new_states.add(v)

for v in pomdp.find_absorbing_states():
    new_states.add(v)

len(new_states)


64

In [16]:
new_rewards = {} # vertex, action -> reward

for v in new_states:
    for action in new_actions:
        r = pomdp.get_reward(v, action)
        if r != 0:
            if v not in new_rewards:
                new_rewards[v] = {}
            new_rewards[v][action] = r

new_obs_f = {} # action, v_to, obs -> prob
for action in new_actions:
    if action not in new_obs_f.keys():
        new_obs_f[action] = {}
    for v in new_states:
        if v not in new_obs_f[action]:
            new_obs_f[action][v] = {}
        found_t_type_err = True
        found_f_type_err = False
        for obs in new_observations:
            prob = pomdp.get_obs_prob(action, v, obs)
            if prob > 0:
                if action in [0,1, 3]:
                    if prob > 0.1:
                        if action in [0, 3]:
                            if prob > 0.7:
                                new_obs_f[action][v][obs] = 0.9
                        else:
                            assert action == 1
                            if prob > 0.4:
                                new_obs_f[action][v][obs] = 0.8
                    else:
                        is_t_type_err = False
                        if action in [0, 3]:
                            if math.isclose(prob, 0.04, rel_tol=1e-4, abs_tol=1e-4):
                                is_t_type_err = True
                            else:
                                assert math.isclose(prob, 0.01, rel_tol=1e-4, abs_tol=1e-4)
                                prob = 0.1
                                is_t_type_err = False
                        else:
                            assert(action == 1)
                            if math.isclose(prob, 0.06, rel_tol=1e-4, abs_tol=1e-4):
                                is_t_type_err = True
                            else:
                                assert math.isclose(prob, 0.04, rel_tol=1e-4, abs_tol=1e-4)
                                is_t_type_err = False
                                prob = 0.2
                        if (not found_t_type_err) and is_t_type_err:
                            found_t_type_err = True
                            new_obs_f[action][v][obs] = prob
                        elif (not found_f_type_err) and (not is_t_type_err):
                            found_f_type_err = True
                            new_obs_f[action][v][obs] = prob

                else:
                    new_obs_f[action][v][obs] = prob
new_obs_f

{0: {0: {1: 0.9},
  1: {1: 0.9},
  2: {1: 0.9},
  3: {1: 0.9},
  4: {2: 0.9, 13: 0.1},
  9: {3: 0.9, 12: 0.1},
  14: {4: 0.9, 13: 0.1},
  19: {5: 0.9, 14: 0.1},
  24: {6: 0.9, 15: 0.1},
  29: {7: 0.9, 16: 0.1},
  34: {8: 0.9, 17: 0.1},
  39: {9: 0.9, 18: 0.1},
  44: {10: 0.9, 19: 0.1},
  49: {11: 0.9, 20: 0.1},
  54: {3: 0.1, 12: 0.9},
  55: {3: 0.1, 12: 0.9},
  56: {3: 0.1, 12: 0.9},
  57: {3: 0.1, 12: 0.9},
  58: {3: 0.1, 12: 0.9},
  59: {2: 0.1, 13: 0.9},
  60: {2: 0.1, 13: 0.9},
  61: {2: 0.1, 13: 0.9},
  62: {2: 0.1, 13: 0.9},
  63: {2: 0.1, 13: 0.9},
  64: {3: 0.1, 14: 0.9},
  65: {3: 0.1, 14: 0.9},
  66: {3: 0.1, 14: 0.9},
  67: {3: 0.1, 14: 0.9},
  68: {3: 0.1, 14: 0.9},
  69: {4: 0.1, 15: 0.9},
  70: {4: 0.1, 15: 0.9},
  71: {4: 0.1, 15: 0.9},
  72: {4: 0.1, 15: 0.9},
  73: {4: 0.1, 15: 0.9},
  74: {5: 0.1, 16: 0.9},
  75: {5: 0.1, 16: 0.9},
  76: {5: 0.1, 16: 0.9},
  77: {5: 0.1, 16: 0.9},
  78: {5: 0.1, 16: 0.9},
  79: {6: 0.1, 17: 0.9},
  80: {6: 0.1, 17: 0.9},
  81: {6: 0.

In [17]:
def is_friend_state(state):
    return state < 54

def is_enemy_state(state):
    return not is_friend_state(state)

rewards_d = pomdp.enemy_states_d
rewards_p = list(rewards_d.keys())
distance_d = {}

for (index, r_p) in enumerate(rewards_p):
    distance_d[index] = set()
    for v in rewards_d[r_p]:
        distance_d[index].add(v)
visibility_d = {}
for i in range(0, 5):
    visibility_d[i] = set()

for (d, vertices) in distance_d.items():
    if d > 0:
        vs_sorted = sort_by_visibility(vertices)
        assert(len(vs_sorted) > 0)
        for (index, v) in enumerate(vs_sorted):
            visibility_d[index].add(v)

def get_visibility(state):
    for (k, vs) in visibility_d.items():
        if state in vs:
            return k

    return 100000

In [18]:
new_trans_f = {} # trans_f: Dict[int, Dict[int, Dict[int, float]]]

for state in new_states:
    for action in new_actions:
        for state_ in new_states:
            should_add = False
            if is_friend_state(state):
                should_add = True
            else:
                assert(is_enemy_state(state))
                v_s = get_visibility(state)
                v_s_ = get_visibility(state_)
                if v_s <= v_s_:
                    should_add = True
            if should_add:
                prob = pomdp.get_trans(state, action, state_)
                if prob > 0:
                    if state not in new_trans_f.keys():
                        new_trans_f[state] = {}
                    if action not in new_trans_f[state].keys():
                        new_trans_f[state][action] = {}
                    new_trans_f[state][action][state_] = prob

new_trans_f

{0: {0: {0: 1.0}, 1: {0: 1.0}, 2: {0: 1.0}, 3: {0: 1.0}},
 1: {0: {1: 1.0}, 1: {1: 1.0}, 2: {1: 1.0}, 3: {1: 1.0}},
 2: {0: {2: 1.0}, 1: {2: 1.0}, 2: {2: 1.0}, 3: {2: 1.0}},
 3: {0: {3: 1.0}, 1: {3: 1.0}, 2: {3: 1.0}, 3: {3: 1.0}},
 4: {0: {0: 1.0}, 1: {0: 1.0}, 2: {0: 1.0}, 3: {3: 1.0}},
 9: {0: {4: 0.04, 9: 0.01},
  1: {4: 0.72, 9: 0.18},
  2: {4: 0.8, 9: 0.2},
  3: {3: 0.81, 4: 0.0304, 9: 0.0076}},
 14: {0: {9: 0.04, 14: 0.01},
  1: {9: 0.72, 14: 0.18},
  2: {9: 0.8, 14: 0.2},
  3: {3: 0.64, 9: 0.0576, 14: 0.0144}},
 19: {0: {14: 0.04, 19: 0.01},
  1: {14: 0.72, 19: 0.18},
  2: {14: 0.8, 19: 0.2},
  3: {3: 0.49, 14: 0.0816, 19: 0.0204}},
 24: {0: {19: 0.04, 24: 0.01},
  1: {19: 0.72, 24: 0.18},
  2: {19: 0.8, 24: 0.2},
  3: {3: 0.36, 19: 0.1024, 24: 0.0256}},
 29: {0: {24: 0.04, 29: 0.01},
  1: {24: 0.72, 29: 0.18},
  2: {24: 0.8, 29: 0.2},
  3: {3: 0.25, 24: 0.12, 29: 0.03}},
 34: {0: {29: 0.04, 34: 0.01},
  1: {29: 0.72, 34: 0.18},
  2: {29: 0.8, 34: 0.2},
  3: {3: 0.16, 29: 0.134

In [19]:
new_pomdp = POMDP(is_empty=True)
new_pomdp.states = new_states
new_pomdp.actions = new_actions
new_pomdp.trans_f = new_trans_f
new_pomdp.observations = new_observations
new_pomdp.obs_f = new_obs_f
new_pomdp.rewards = new_rewards

In [20]:
old_to_new_s = new_pomdp.normalize()

In [21]:
def get_enemy(distance, visibility):
    s1 = distance_d[distance]
    s2 = visibility_d[visibility]
    s = s1.intersection(s2)
    assert(len(s) == 1)
    for e in s:
        return old_to_new_s[e]

    assert(False)

def get_friend(distance):
    friend_rewards = list(pomdp.friend_states_d.keys())

    for (index, r_p) in enumerate(friend_rewards):
        if index == distance:
            result = min(pomdp.friend_states_d[r_p])
            assert(result in new_pomdp.states)
            return old_to_new_s[result]

    assert(False)

visibilities = [0, 2, 4]

for d1 in range(1, 4):
    for d2 in range(d1+1, 4):
        for v1 in visibilities:
            for v2 in visibilities:
                if v1 != v2:
                    e1 = get_enemy(d1, v1)
                    e2 = get_enemy(d2, v2)
                    f1 = get_friend(d2)
                    new_pomdp.write_test_case({e1, e2, f1}, f"iff_{d1}_{d2}_{v1}_{v2}")



In [22]:
f = open("helper_iff_slurm.sh", "w")
for d1 in range(1, 4):
    for d2 in range(d1+1, 4):
        for v1 in visibilities:
            for v2 in visibilities:
                if v1 != v2:
                    e1 = get_enemy(d1, v1)
                    e2 = get_enemy(d2, v2)
                    f.write(f"sbatch f1_slurm.sh iff_{d1}_{d2}_{v1}_{v2}\n")

f.close()